Script ini melakukan preprocessing pada dataset ulasan OVO yang sudah dilabeli
sebelum digunakan untuk analisis sentimen atau training model machine learning.

Preprocessing steps:
1. Case folding (lowercase)
2. Pemisahan emoji
3. Penghapusan angka dan tanda baca
4. Normalisasi kata slang
5. Tokenisasi
6. Penghapusan stopwords
7. Stemming

Pada tahap preprocessing, penelitian ini menggunakan sumber daya NLP Bahasa Indonesia dari repositori yang dikembangkan oleh [Louis Owen](https://github.com/louisowen6/NLP_bahasa_resources).

Normalisasi kata tidak baku dilakukan melalui modul Python slang_in_df_ovo.py, yang menggabungkan daftar slang dari file combined_root_words.txt dengan tambahan slang dari dataset.

Stopwords dihapus menggunakan kombinasi daftar dari library Sastrawi, file combined_stop_words.txt, serta tambahan dari dataset.

Daftar kata dasar digunakan sebagai acuan untuk mendeteksi dan menormalisasi kata tidak baku secara iteratif hingga sebagian besar kata menjadi bentuk baku.

# IMPORT LIBRARIES

In [25]:
import pickle
import string
from typing import List, Set

import emoji
import nltk
import pandas as pd
from nltk import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from tqdm import tqdm

# Custom modules
import slang_in_df_ovo

# DOWNLOAD NLTK RESOURCES

In [26]:
# Mengunduh model tokenizer NLTK yang diperlukan oleh fungsi word_tokenize
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# Aktifkan integrasi tqdm dengan pandas untuk progress bar
tqdm.pandas()

# HELPER FUNCTIONS

In [27]:
def separate_emoji(text: str) -> str:
    """
    Memisahkan emoji dari teks agar menjadi token terpisah.

    Args:
        text: String teks yang mungkin mengandung emoji

    Returns:
        String dengan emoji yang dipisahkan oleh spasi
    """
    return emoji.replace_emoji(text, replace=lambda e, data: f" {e} ")


def load_root_words(filepath: str = "combined_root_words.txt") -> Set[str]:
    """
    Memuat daftar kata dasar dari file.

    Args:
        filepath: Path ke file berisi kata dasar

    Returns:
        Set berisi kata-kata dasar
    """
    with open(filepath, "r", encoding="utf-8") as file:
        words = [line.strip() for line in file if line.strip()]
    return set(words)


def load_stopwords(filepath: str = "combined_stop_words.txt") -> Set[str]:
    """
    Memuat daftar stopwords dari file.

    Args:
        filepath: Path ke file berisi stopwords

    Returns:
        Set berisi stopwords
    """
    with open(filepath, "r", encoding="utf-8") as file:
        words = [line.strip() for line in file if line.strip()]
    return set(words)


# Cache untuk menyimpan hasil stemming agar tidak perlu mengulang proses
stem_cache = {}

def get_stemmed_term(term: str, stemmer) -> str:
    """
    Melakukan stemming pada sebuah term dengan cache.

    Args:
        term: Kata yang akan di-stem
        stemmer: Object stemmer Sastrawi

    Returns:
        Kata hasil stemming
    """
    if term not in stem_cache:
        stem_cache[term] = stemmer.stem(term)
    return stem_cache[term]

# LOAD RESOURCES

In [28]:
# Memuat kata dasar
listKataDasar = load_root_words("combined_root_words.txt")
print(f"✓ Loaded {len(listKataDasar)} root words")

# Memuat stopwords awal
listOfStopwords = load_stopwords("combined_stop_words.txt")
print(f"✓ Loaded {len(listOfStopwords)} initial stopwords")

✓ Loaded 28010 root words
✓ Loaded 673 initial stopwords


# LOAD DATASET

In [29]:
# Membaca dataset hasil scraping/review yang sudah diberi label sentimen
df = pd.read_csv('/content/ulasan_ovo_labeled_final.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Dataset shape: (20000, 3)
Columns: ['reviewId', 'content', 'label_final']


In [31]:
# Menampilkan sample data
print("\nSample data (before preprocessing):")
df.sample(5, random_state=12)


Sample data (before preprocessing):


,reviewId,content,label_final
15958,d3582c90-1c45-4d2b-a9e8-34c7764e8f30,PARAH!!! untuk melakukan transaksi harus upgra...,Negatif
19264,c34638b8-15f9-42a7-bc65-af05471edfad,"tolong ya kepada ovo ,, proses pembelian pulsa...",Negatif
17996,23ef8407-3df9-45b3-8071-e63e12a82785,aku udah lama menggunakan ovo tapi setelah upd...,Negatif
17179,fd6b7e3d-ba87-475e-a576-5bb78a2dee76,kenapa saldoku ke top up ke itemku dan terpoto...,Negatif
3722,fbc4d1a0-99d1-4277-aa9d-2ce1751c46ac,mohon dan tolong. perbaikan dan penyempurnaan ...,Negatif


# PREPROCESSING PIPELINE

## Step 1: Case Folding - Mengubah semua teks menjadi lowercase

In [32]:
print("\n[Step 1] Case folding (lowercasing)...")
df['content'] = df['content'].str.lower()
print("✓ Completed")


[Step 1] Case folding (lowercasing)...
✓ Completed


## Step 2: Emoji Separation - Memisahkan emoji dari teks

In [33]:
print("\n[Step 2] Separating emojis...")
df["content"] = df["content"].apply(separate_emoji)
print("✓ Completed")


[Step 2] Separating emojis...
✓ Completed


## Step 3: Clean Text - Menghapus angka, tanda baca, dan whitespace berlebih

In [34]:
print("\n[Step 3] Cleaning text (remove digits, punctuation, extra spaces)...")
df["content"] = (
    df["content"]
    .str.replace(r"\d+", "", regex=True)                      # Menghapus angka
    .str.replace(f"[{string.punctuation}]", "", regex=True)    # Menghapus tanda baca
    .str.replace(r"\s+", " ", regex=True)                      # Multiple whitespace -> single space
    .str.strip()                                               # Trim whitespace
)
print("✓ Completed")


[Step 3] Cleaning text (remove digits, punctuation, extra spaces)...
✓ Completed


## Step 4: Slang Normalization - Mengubah kata tidak baku menjadi baku

In [35]:
print("\n[Step 4] Normalizing slang words...")
df["content"] = df["content"].apply(slang_in_df_ovo.fix_slangwords)
print("✓ Completed")


[Step 4] Normalizing slang words...
✓ Completed


## Step 5: Tokenization - Memecah teks menjadi token-token kata

In [36]:
print("\n[Step 5] Tokenizing text...")
df["content"] = df["content"].apply(word_tokenize)
print("✓ Completed")


[Step 5] Tokenizing text...
✓ Completed


## Step 6: Stopwords Removal - Menghapus kata-kata stopword

In [37]:
print("\n[Step 6] Removing stopwords...")

# Menambahkan stopword dari Sastrawi
stopwords_factory = StopWordRemoverFactory()
listOfStopwords.update(stopwords_factory.get_stop_words())

# Menambahkan stopword kustom yang sering muncul pada dataset OVO
custom_stopwords = {
    "apk", "aplikasi", "aplikasinya", "app", "bahkan",
    "ovo", "banking", "nya", "sejak", "sih",
    "warahmatullahi", "wabarakatuh"
}
listOfStopwords.update(custom_stopwords)

print(f"Total stopwords after addition: {len(listOfStopwords)}")

# Menghapus stopwords dari setiap token
df["content"] = df["content"].apply(
    lambda tokens: [t for t in tokens if t not in listOfStopwords]
)
print("✓ Completed")


[Step 6] Removing stopwords...
Total stopwords after addition: 690
✓ Completed


## Step 7: Remove Empty Rows - Menghapus baris tanpa token

In [38]:
print("\n[Step 7] Removing empty rows...")
initial_rows = len(df)
df = df[df["content"].str.len() > 0]
removed_rows = initial_rows - len(df)
print(f"✓ Removed {removed_rows} rows with no tokens")


[Step 7] Removing empty rows...
✓ Removed 727 rows with no tokens


## Step 8: Stemming - Mengubah kata menjadi bentuk dasarnya

In [39]:
print("\n[Step 8] Stemming tokens...")

# Inisialisasi stemmer
stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

# Melakukan stemming dengan progress bar
df["content"] = df["content"].progress_apply(
    lambda tokens: [get_stemmed_term(t, stemmer) for t in tokens]
)
print(f"✓ Completed. Cache size: {len(stem_cache)} terms")


[Step 8] Stemming tokens...


100%|██████████| 19273/19273 [34:25<00:00,  9.33it/s]

✓ Completed. Cache size: 15923 terms


# FINAL RESULT

In [40]:
print(f"\nFinal dataset shape: {df.shape}")


Final dataset shape: (19273, 3)


In [41]:
print("\nSample of preprocessed data:")
df.sample(5, random_state=12)


Sample of preprocessed data:


,reviewId,content,label_final
9627,2fce489d-571d-4485-828f-0b3f8928cc08,[top],Positif
10244,e82cd784-b066-4c83-b7da-f248a2764185,"[kasih, bintang, tingkat, layan, performa]",Positif
1281,5b7dc5d5-4afb-4064-a3b4-b6fb93d075a1,"[bagus, geng]",Positif
16126,c7d2c38f-e4c2-4c2e-8b77-0ae11890a252,"[masuk, akun, setela, isi, saldo]",Netral
16776,097377ab-32fa-4958-ac2c-c4a8d0239361,"[blokir, susah, banget, , , , , , , ]",Negatif


In [42]:
print("\nSample tokenized review (first 3 rows):")
for i, tokens in enumerate(df["content"].head(3)):
    print(f"Review {i+1}: {tokens[:10]}..." if len(tokens) > 10 else f"Review {i+1}: {tokens}")


Sample tokenized review (first 3 rows):
Review 1: ['upgrade', 'kamera', 'buka', 'pakai', 'hp', 'ram', 'gb']
Review 2: ['mantap']
Review 3: ['cuus']


# SAVE PREPROCESSED DATA

In [43]:
# Uncomment to save the preprocessed data
df.to_csv('/content/ulasan_ovo_preprocessed.csv', index=False)
print("\n✓ Preprocessed data saved to 'ulasan_ovo_preprocessed.csv'")


✓ Preprocessed data saved to 'ulasan_ovo_preprocessed.csv'


In [44]:
# Save stem cache for later use (optional)
with open('stem_cache.pkl', 'wb') as f:
  pickle.dump(stem_cache, f)
print("✓ Stem cache saved to 'stem_cache.pkl'")

✓ Stem cache saved to 'stem_cache.pkl'
